# Train Age + Gender (FairFace)

**Dataset:** FairFace (tai thang vao local Colab)  
**Backbone:** EfficientNetB2  
**Strategy:** 2-phase fine-tuning + class weights  
**Output:** gender_model.keras + age_model.keras

In [ ]:
print('=== [1/10] Cai dat thu vien ===')
!pip -q install kaggle tensorflow pandas scikit-learn matplotlib seaborn
print('Da cai dat xong!')

In [ ]:
print('=== [2/10] Mount Google Drive (chi de luu model) ===')
from google.colab import drive
drive.mount('/content/drive')
print('Da mount Drive!')

In [ ]:
print('=== [3/10] Tai dataset FairFace THANG vao local Colab ===')
import os
import subprocess
import time
from pathlib import Path
from google.colab import files

# ====== CAU HINH ======
FAIRFACE_SLUG = 'aibloy/fairface'
DATASET_DIR   = Path('/content/fairface')  # Local SSD - nhanh!
MODELS_DIR    = Path('/content/drive/MyDrive/face_age_gender_emotion/models')  # Luu model len Drive
# =====================

IMAGE_SIZE = 224
BATCH_SIZE = 64
EPOCHS_PHASE1 = 8
EPOCHS_PHASE2 = 25
MAX_TRAIN_SAMPLES = None

MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Kiem tra da co tren local chua
has_local = DATASET_DIR.exists() and any(DATASET_DIR.rglob('*.csv'))

if has_local:
    print(f'Dataset da co tai {DATASET_DIR}, bo qua.')
else:
    print('Upload file kaggle.json (~1KB):')
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise FileNotFoundError('Chua upload kaggle.json!')

    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    (kaggle_dir / 'kaggle.json').write_bytes(uploaded['kaggle.json'])
    os.chmod(kaggle_dir / 'kaggle.json', 0o600)
    print('Kaggle API OK!')

    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    print(f'\nDang tai {FAIRFACE_SLUG} thang vao local Colab...')
    print('(Tai thang vao SSD local, KHONG qua Drive, nhanh hon nhieu!)')
    start = time.time()
    subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', FAIRFACE_SLUG,
        '-p', str(DATASET_DIR),
        '--unzip'
    ], check=True)
    print(f'Tai + giai nen xong trong {time.time()-start:.0f}s!')

# Tim thu muc chua CSV
def find_root_with_csv(root):
    if any(root.glob('*.csv')):
        return root
    for sub in sorted(root.iterdir()):
        if sub.is_dir() and any(sub.glob('*.csv')):
            return sub
        if sub.is_dir():
            for sub2 in sub.iterdir():
                if sub2.is_dir() and any(sub2.glob('*.csv')):
                    return sub2
    return root

DATASET_DIR = find_root_with_csv(DATASET_DIR)
print(f'\nDATASET_DIR: {DATASET_DIR}')
print(f'MODELS_DIR:  {MODELS_DIR}')

# Hien cau truc
for item in sorted(DATASET_DIR.iterdir()):
    if item.is_dir():
        count = sum(1 for f in item.rglob('*') if f.is_file())
        print(f'  {item.name}/  ({count} files)')
    else:
        print(f'  {item.name}  ({item.stat().st_size/(1024*1024):.1f} MB)')
print('OK!')

In [ ]:
print('=== [4/10] Doc CSV ===')
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

AGE_LABELS = ['0-2', '3-9', '10-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70+']
GENDER_LABELS = ['Female', 'Male']
NUM_AGE = len(AGE_LABELS)
NUM_GENDER = len(GENDER_LABELS)

AGE_TO_ID = {
    '0-2': 0, '3-9': 1, '10-19': 2, '20-29': 3, '30-39': 4,
    '40-49': 5, '50-59': 6, '60-69': 7, '70+': 8, 'more than 70': 8
}
GENDER_TO_ID = {'Female': 0, 'Male': 1, 'female': 0, 'male': 1, 'F': 0, 'M': 1}

def find_csv(root, names):
    names = {n.lower() for n in names}
    for p in root.rglob('*.csv'):
        if p.name.lower() in names:
            return p
    return None

train_csv = find_csv(DATASET_DIR, ['fairface_label_train.csv', 'train_labels.csv', 'train.csv'])
val_csv = find_csv(DATASET_DIR, ['fairface_label_val.csv', 'val_labels.csv', 'val.csv'])

if train_csv is None:
    print('CSV files:')
    for f in DATASET_DIR.rglob('*.csv'):
        print(f'  {f}')
    raise FileNotFoundError('Khong tim thay train CSV!')

print(f'Train CSV: {train_csv}')
print(f'Val CSV:   {val_csv}')

train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(val_csv) if val_csv else None
print(f'Train rows: {len(train_df)}')
print(f'Columns: {list(train_df.columns)}')
train_df.head(3)

In [ ]:
print('=== [5/10] Xu ly data + class weights ===')
import time
t0 = time.time()

def normalize_df(df):
    path_col = next((c for c in ['file', 'path', 'image', 'filename'] if c in df.columns), None)
    if path_col is None:
        raise ValueError(f'Khong tim thay cot anh, columns={list(df.columns)}')
    if 'age' not in df.columns or 'gender' not in df.columns:
        raise ValueError('Thieu cot age hoac gender')

    out = df[[path_col, 'age', 'gender']].copy()
    out.columns = ['path', 'age', 'gender']
    out['image_path'] = out['path'].apply(
        lambda p: str(DATASET_DIR / str(p).strip().replace('\\\\', '/'))
    )
    out['age_id'] = out['age'].astype(str).str.strip().map(AGE_TO_ID)
    out['gender_id'] = out['gender'].astype(str).str.strip().map(GENDER_TO_ID)
    out = out.dropna(subset=['age_id', 'gender_id'])
    out['age_id'] = out['age_id'].astype('int32')
    out['gender_id'] = out['gender_id'].astype('int32')
    return out[['image_path', 'gender_id', 'age_id']]

print('Xu ly train...')
train_data = normalize_df(train_df)
print(f'  Train: {len(train_data)} ({time.time()-t0:.1f}s)')

if val_df is not None:
    val_data = normalize_df(val_df)
    print(f'  Val: {len(val_data)}')
else:
    val_data = None

if val_data is None or val_data.empty:
    print('Chia train/val (85/15)...')
    train_data, val_data = train_test_split(
        train_data, test_size=0.15, random_state=42, stratify=train_data['age_id']
    )

# Kiem tra file mau
sample = train_data['image_path'].iloc[0]
print(f'\nFile mau: {sample}')
print(f'Ton tai: {Path(sample).exists()}')

if MAX_TRAIN_SAMPLES:
    train_data = train_data.sample(MAX_TRAIN_SAMPLES, random_state=42)

# Class weights
age_unique = np.sort(np.unique(train_data['age_id'].values))
age_weights = compute_class_weight('balanced', classes=age_unique, y=train_data['age_id'].values)
age_weight_dict = dict(zip(age_unique.astype(int), age_weights))

gender_unique = np.sort(np.unique(train_data['gender_id'].values))
gender_weights = compute_class_weight('balanced', classes=gender_unique, y=train_data['gender_id'].values)
gender_weight_dict = dict(zip(gender_unique.astype(int), gender_weights))

print(f'\nTrain: {len(train_data)} | Val: {len(val_data)}')
print('\nAge:')
for idx, cnt in train_data['age_id'].value_counts().sort_index().items():
    print(f'  {AGE_LABELS[idx]:>6s}: {cnt:>5d} | w={age_weight_dict.get(idx,0):.3f}')
print('\nGender:')
for idx, cnt in train_data['gender_id'].value_counts().sort_index().items():
    print(f'  {GENDER_LABELS[idx]:>6s}: {cnt:>5d} | w={gender_weight_dict.get(idx,0):.3f}')
print(f'\nXong! ({time.time()-t0:.1f}s)')

In [ ]:
print('=== [6/10] Tao tf.data pipeline ===')
AUTOTUNE = tf.data.AUTOTUNE
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print(f'Mixed precision: {tf.keras.mixed_precision.global_policy().name}')

def load_image(path, gender_id, age_id, training=False):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    if training:
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_brightness(image, 0.15)
        image = tf.image.random_contrast(image, 0.85, 1.15)
        image = tf.image.random_saturation(image, 0.85, 1.15)
        image = tf.clip_by_value(image, 0.0, 1.0)
    return image, {'gender': gender_id, 'age': age_id}

def make_dataset(df, training):
    ds = tf.data.Dataset.from_tensor_slices((
        df['image_path'].values.astype(str),
        df['gender_id'].values.astype('int32'),
        df['age_id'].values.astype('int32')
    ))
    if training:
        ds = ds.shuffle(min(len(df), 10000), seed=42, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, g, a: load_image(p, g, a, training), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_data, training=True)
val_ds = make_dataset(val_data, training=False)

print(f'Train batches: {tf.data.experimental.cardinality(train_ds).numpy()}')
print(f'Val batches:   {tf.data.experimental.cardinality(val_ds).numpy()}')
print('Pipeline OK!')

In [ ]:
print('=== [7/10] Phase 1: Freeze backbone, train heads ===')
from tensorflow import keras
from tensorflow.keras import layers

inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), name='image')
base = keras.applications.EfficientNetB2(
    include_top=False, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), weights='imagenet'
)
base.trainable = False
print(f'Backbone: EfficientNetB2 (FROZEN, {len(base.layers)} layers)')

x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D(name='gap')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.30)(x)

g = layers.Dense(64, activation='relu', name='gender_fc')(x)
g = layers.Dense(NUM_GENDER, name='gender_logits')(g)
gender_output = layers.Activation('softmax', dtype='float32', name='gender')(g)

a = layers.Dense(128, activation='relu', name='age_fc')(x)
a = layers.Dense(NUM_AGE, name='age_logits')(a)
age_output = layers.Activation('softmax', dtype='float32', name='age')(a)

model = keras.Model(inputs=inputs, outputs={'gender': gender_output, 'age': age_output})
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss={'gender': keras.losses.SparseCategoricalCrossentropy(), 'age': keras.losses.SparseCategoricalCrossentropy()},
    metrics={'gender': ['accuracy'], 'age': ['accuracy']},
)

trainable = sum(p.numpy().size for p in model.trainable_weights)
print(f'Trainable: {trainable:,}')
print(f'Training {EPOCHS_PHASE1} epochs...')

history1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_PHASE1)
print(f'\nPhase 1 xong! Gender acc: {history1.history["val_gender_accuracy"][-1]:.4f}, Age acc: {history1.history["val_age_accuracy"][-1]:.4f}')

In [ ]:
print('=== [8/10] Phase 2: Unfreeze backbone, fine-tune ===')
base.trainable = True
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-5),
    loss={'gender': keras.losses.SparseCategoricalCrossentropy(), 'age': keras.losses.SparseCategoricalCrossentropy()},
    metrics={'gender': ['accuracy'], 'age': ['accuracy']},
)
print(f'Backbone UNFROZEN | Trainable: {sum(p.numpy().size for p in model.trainable_weights):,}')
print(f'Training {EPOCHS_PHASE2} epochs, lr=3e-5...')

save_path = str(MODELS_DIR / 'age_gender_best.keras')
callbacks = [
    keras.callbacks.ModelCheckpoint(filepath=save_path, monitor='val_loss', save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),
]

history2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_PHASE2, callbacks=callbacks,
)
print(f'\nPhase 2 xong!')

In [ ]:
print('=== [9/10] Evaluation ===')
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

full_history = {}
for key in history1.history:
    full_history[key] = history1.history[key] + history2.history.get(key, [])
with open(MODELS_DIR / 'age_gender_history.json', 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in full_history.items()}, f)
print('Da luu history.')

print('Predicting...')
y_gender_true, y_age_true = [], []
y_gender_pred, y_age_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_gender_true.extend(labels['gender'].numpy())
    y_age_true.extend(labels['age'].numpy())
    y_gender_pred.extend(np.argmax(preds['gender'], axis=1))
    y_age_pred.extend(np.argmax(preds['age'], axis=1))

print(f'\n{"="*50}')
print('GENDER')
print(f'{"="*50}')
print(classification_report(y_gender_true, y_gender_pred, target_names=GENDER_LABELS))

print(f'{"="*50}')
print('AGE')
print(f'{"="*50}')
print(classification_report(y_age_true, y_age_pred, target_names=AGE_LABELS))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, yt, yp, labels, title in [
    (axes[0], y_gender_true, y_gender_pred, GENDER_LABELS, 'Gender'),
    (axes[1], y_age_true, y_age_pred, AGE_LABELS, 'Age'),
]:
    cm = confusion_matrix(yt, yp)
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, ax=ax, cmap='Blues')
    ax.set_title(f'{title} Confusion Matrix')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(str(MODELS_DIR / 'age_gender_confusion.png'), dpi=150)
plt.show()
print('Da luu confusion matrix.')

In [ ]:
print('=== [10/10] Tach va luu model ===')

gender_model = keras.Model(inputs=model.input, outputs=model.get_layer('gender').output, name='gender_model')
age_model = keras.Model(inputs=model.input, outputs=model.get_layer('age').output, name='age_model')

gender_path = MODELS_DIR / 'gender_model.keras'
age_path = MODELS_DIR / 'age_model.keras'

gender_model.save(gender_path)
age_model.save(age_path)

print(f'gender_model: {gender_path} ({gender_path.stat().st_size/(1024*1024):.1f} MB)')
print(f'age_model:    {age_path} ({age_path.stat().st_size/(1024*1024):.1f} MB)')

print(f'\nTat ca files trong models/:')
for f in sorted(MODELS_DIR.iterdir()):
    if not f.name.startswith('.'):
        print(f'  {f.name:40s} {f.stat().st_size/(1024*1024):.1f} MB')

print(f'\nHOAN TAT! Copy gender_model.keras va age_model.keras ve may local.')